# Scout 01 — Ingest Euro 2024 (StatsBomb open data)
Loads all 51 matches -> Cloud Storage (raw archive) -> BigQuery `statsbomb_raw`.
Run top to bottom. Edit the CONFIG cell first.

In [ ]:
# CONFIG — edit these two lines
PROJECT_ID = "statsbomb-football-iq"
BUCKET     = "statsbomb-football-iq-raw"   # the bucket from Part 0
COMPETITION_ID = 55   # UEFA Euro
SEASON_ID      = 282  # 2024

In [ ]:
!pip install -q statsbombpy pandas pyarrow google-cloud-bigquery google-cloud-storage db-dtypes
from google.colab import auth
auth.authenticate_user()   # log in with the SAME google account as your GCP project
print("authed")

In [ ]:
from statsbombpy import sb
import pandas as pd, json, warnings
warnings.filterwarnings("ignore")

# Match list for the whole tournament
matches = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
print("matches found:", len(matches))
match_ids = matches["match_id"].tolist()
matches.head()

In [ ]:
# Pull events + lineups + 360 for every match. Takes a few minutes.
all_events, all_lineups, all_360 = [], [], []
for i, mid in enumerate(match_ids, 1):
    ev = sb.events(match_id=mid); ev["match_id"] = mid
    all_events.append(ev)
    try:
        fr = sb.frames(match_id=mid, fmt="dataframe"); fr["match_id"] = mid
        all_360.append(fr)
    except Exception as e:
        print("no 360 for", mid)
    print(f"{i}/{len(match_ids)} match {mid}: {len(ev)} events")

events_df = pd.concat(all_events, ignore_index=True)
frames_df = pd.concat(all_360, ignore_index=True) if all_360 else pd.DataFrame()
print("TOTAL events:", len(events_df), "| 360 rows:", len(frames_df))

In [ ]:
# Flatten nested dict columns to JSON strings so BigQuery accepts them.
def stringify(df):
    df = df.copy()
    for c in df.columns:
        if df[c].apply(lambda v: isinstance(v,(dict,list))).any():
            df[c] = df[c].apply(lambda v: json.dumps(v) if isinstance(v,(dict,list)) else v)
    df.columns = [c.replace(".","_").replace(" ","_") for c in df.columns]
    return df

events_df  = stringify(events_df)
matches_df = stringify(matches)
frames_df  = stringify(frames_df) if not frames_df.empty else frames_df

In [ ]:
# 1) Archive raw JSON to Cloud Storage (immutable source of truth)
from google.cloud import storage
gcs = storage.Client(project=PROJECT_ID)
b = gcs.bucket(BUCKET)
b.blob("raw/events.json").upload_from_string(events_df.to_json(orient="records"))
b.blob("raw/matches.json").upload_from_string(matches_df.to_json(orient="records"))
if not frames_df.empty:
    b.blob("raw/frames360.json").upload_from_string(frames_df.to_json(orient="records"))
print("archived to gs://%s/raw/" % BUCKET)

In [ ]:
# 2) Load into BigQuery statsbomb_raw
from google.cloud import bigquery
bq = bigquery.Client(project=PROJECT_ID)
bq.create_dataset(bigquery.Dataset(f"{PROJECT_ID}.statsbomb_raw"), exists_ok=True)
cfg = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE", autodetect=True)

bq.load_table_from_dataframe(events_df,  f"{PROJECT_ID}.statsbomb_raw.events_raw",  job_config=cfg).result()
bq.load_table_from_dataframe(matches_df, f"{PROJECT_ID}.statsbomb_raw.matches_raw", job_config=cfg).result()
if not frames_df.empty:
    bq.load_table_from_dataframe(frames_df, f"{PROJECT_ID}.statsbomb_raw.frames360_raw", job_config=cfg).result()
print("loaded to BigQuery statsbomb_raw. INGEST COMPLETE.")